# YouTube Video Q&A Assistant — Notebook thử nghiệm

Notebook này demo pipeline RAG cơ bản với **LlamaIndex + Chroma + YouTube Transcript API**:

1. Lấy transcript từ video YouTube  
2. Đóng gói transcript thành `Document` có `metadata`  
3. Chunk bằng `SentenceSplitter`  
4. Tạo embedding và lưu vào ChromaDB  
5. Truy vấn các đoạn liên quan  
6. Tùy chọn: dùng LLM để sinh câu trả lời

> Notebook này ưu tiên chạy được để **kiểm tra retrieval trước**, chưa bắt buộc phải có API key.


## 1. Cài thư viện

Chạy cell này nếu bạn chưa cài thư viện.  
Trong VS Code/Jupyter local, sau khi cài xong nên **Restart Kernel**.


In [ ]:
# Nếu chạy lần đầu, bỏ comment dòng dưới:
# !pip install -U llama-index chromadb youtube-transcript-api python-dotenv llama-index-vector-stores-chroma llama-index-embeddings-huggingface sentence-transformers

## 2. Import thư viện

In [1]:
import os
from pathlib import Path
from urllib.parse import urlparse, parse_qs

import chromadb
from youtube_transcript_api import YouTubeTranscriptApi

from llama_index.core import Document, VectorStoreIndex, StorageContext, Settings
from llama_index.core.node_parser import SentenceSplitter
from llama_index.vector_stores.chroma import ChromaVectorStore
from llama_index.embeddings.huggingface import HuggingFaceEmbedding

print("Import thành công!")

d:\Study\HCMUS\My Project\YouTube Video Q&A Assistant\notebooks\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Import thành công!


## 3. Cấu hình video YouTube

Bạn có thể nhập **URL đầy đủ** hoặc chỉ nhập **video ID**.

Ví dụ:

```text
https://www.youtube.com/watch?v=0LCgem1OAcE&t=881s
```

Video ID đúng sẽ là:

```text
0LCgem1OAcE
```


In [2]:
YOUTUBE_INPUT = "https://www.youtube.com/watch?v=0LCgem1OAcE&t=881s"


def extract_video_id(value: str) -> str:
    """Trích video_id từ URL YouTube hoặc trả về chính value nếu đã là ID."""
    value = value.strip()

    # Nếu người dùng đã nhập ID ngắn, ví dụ: 0LCgem1OAcE
    if "youtube.com" not in value and "youtu.be" not in value:
        return value.split("&")[0].strip()

    parsed = urlparse(value)

    if parsed.hostname in ["www.youtube.com", "youtube.com", "m.youtube.com"]:
        query = parse_qs(parsed.query)
        if "v" not in query:
            raise ValueError("URL YouTube không có tham số v=VIDEO_ID")
        return query["v"][0]

    if parsed.hostname == "youtu.be":
        return parsed.path.lstrip("/").split("?")[0]

    raise ValueError("URL YouTube không hợp lệ")


VIDEO_ID = extract_video_id(YOUTUBE_INPUT)
VIDEO_URL = f"https://www.youtube.com/watch?v={VIDEO_ID}"

print("VIDEO_ID:", VIDEO_ID)
print("VIDEO_URL:", VIDEO_URL)

VIDEO_ID: 0LCgem1OAcE
VIDEO_URL: https://www.youtube.com/watch?v=0LCgem1OAcE


## 4. Lấy transcript từ YouTube

Với version mới của `youtube-transcript-api`, cách dùng nên là:

```python
ytt_api = YouTubeTranscriptApi()
transcript = ytt_api.fetch(video_id)
```


In [3]:
def snippet_get(snippet, key: str):
    """Hỗ trợ cả object kiểu mới và dict kiểu cũ."""
    if hasattr(snippet, key):
        return getattr(snippet, key)
    return snippet[key]


def fetch_transcript(video_id: str, languages=("en", "vi")):
    """Lấy transcript. Ưu tiên English/Vietnamese nếu có."""
    ytt_api = YouTubeTranscriptApi()

    try:
        # API mới
        return ytt_api.fetch(video_id, languages=list(languages))
    except TypeError:
        # Một số version có thể không nhận languages ở fetch
        return ytt_api.fetch(video_id)


transcript = fetch_transcript(VIDEO_ID)

print("Số snippet transcript:", len(list(transcript)))
print("5 dòng đầu:")
for snippet in list(transcript)[:5]:
    print("-", snippet_get(snippet, "text"))

Số snippet transcript: 251
5 dòng đầu:
- First question: Could you introduce yourself please?  
- My name is Ariana Kincaid. 
- And whereabouts in the world are you? 
- I am in West Virginia Charleston West Virginia in the United States.
- And is your name really Ariana?


## 5. Biến transcript thành `Document`

Ở đây mình không gom toàn bộ transcript thành 1 `Document`, mà chia theo từng cửa sổ thời gian nhỏ, ví dụ mỗi đoạn khoảng 90 giây.

Lý do:

- Metadata có thể lưu `start_seconds`
- Khi retrieve, mình biết đoạn đó nằm ở phút nào
- Có thể tạo link nhảy thẳng đến đoạn trong video


In [4]:
def seconds_to_mmss(seconds: float) -> str:
    seconds = int(seconds)
    m, s = divmod(seconds, 60)
    return f"{m:02d}:{s:02d}"


def build_documents_from_transcript(transcript, video_id: str, window_seconds: int = 90):
    docs = []
    current_texts = []
    window_start = None
    last_end = None

    for snippet in transcript:
        text = snippet_get(snippet, "text")
        start = float(snippet_get(snippet, "start"))
        duration = float(snippet_get(snippet, "duration"))
        end = start + duration

        if window_start is None:
            window_start = start

        # Nếu vượt quá window_seconds thì đóng gói thành Document mới
        if current_texts and (start - window_start >= window_seconds):
            start_int = int(window_start)
            docs.append(
                Document(
                    text=" ".join(current_texts),
                    metadata={
                        "source": "youtube",
                        "video_id": video_id,
                        "url": f"https://www.youtube.com/watch?v={video_id}&t={start_int}s",
                        "start_seconds": start_int,
                        "start_time": seconds_to_mmss(start_int),
                        "end_seconds": int(last_end) if last_end is not None else start_int,
                    },
                )
            )
            current_texts = []
            window_start = start

        current_texts.append(text)
        last_end = end

    # Đoạn cuối
    if current_texts:
        start_int = int(window_start)
        docs.append(
            Document(
                text=" ".join(current_texts),
                metadata={
                    "source": "youtube",
                    "video_id": video_id,
                    "url": f"https://www.youtube.com/watch?v={video_id}&t={start_int}s",
                    "start_seconds": start_int,
                    "start_time": seconds_to_mmss(start_int),
                    "end_seconds": int(last_end) if last_end is not None else start_int,
                },
            )
        )

    return docs


documents = build_documents_from_transcript(transcript, VIDEO_ID, window_seconds=90)

print("Số Document tạo được:", len(documents))
print("\nVí dụ metadata Document đầu tiên:")
print(documents[0].metadata)
print("\nNội dung đầu tiên:")
print(documents[0].text[:500])

Số Document tạo được: 17

Ví dụ metadata Document đầu tiên:
{'source': 'youtube', 'video_id': '0LCgem1OAcE', 'url': 'https://www.youtube.com/watch?v=0LCgem1OAcE&t=0s', 'start_seconds': 0, 'start_time': '00:00', 'end_seconds': 90}

Nội dung đầu tiên:
First question: Could you introduce yourself please?   My name is Ariana Kincaid.  And whereabouts in the world are you?  I am in West Virginia Charleston West Virginia in the United States. And is your name really Ariana? It really is.  You're listening to CrowdScience on the BBC World Service.   I'm Caroline Steel and I don't normally question 
everything people say. Could you explain to   listeners why I'm doubting what your real 
name is?  Because I have been in and judged liars contests.  A


## 6. Cấu hình Embedding Model và Splitter

Mặc định LlamaIndex có thể dùng OpenAI embedding, nhưng như vậy cần API key.

Ở đây mình dùng local embedding model từ Hugging Face:

```python
BAAI/bge-small-en-v1.5
```

Model này phù hợp để demo tiếng Anh. Nếu video tiếng Việt, retrieval vẫn có thể chạy nhưng chất lượng có thể chưa tối ưu.


In [5]:
# Embedding local, không cần OpenAI API key
Settings.embed_model = HuggingFaceEmbedding(
    model_name="BAAI/bge-small-en-v1.5"
)

# Không cấu hình LLM ở bước này, vì trước mắt mình test retrieval
# Nếu không set llm, một số query_engine có thể cố gọi OpenAI mặc định.
# Do đó phần chính bên dưới sẽ dùng retriever trước.
Settings.llm = None

splitter = SentenceSplitter(
    chunk_size=512,
    chunk_overlap=50
)

print("Cấu hình embedding và splitter xong.")

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 9503.53it/s]


LLM is explicitly disabled. Using MockLLM.
Cấu hình embedding và splitter xong.


## 7. Tạo Chroma Vector Store

Dữ liệu vector sẽ được lưu ở thư mục:

```text
./chroma_db
```

Nếu muốn build lại từ đầu, bạn có thể xóa thư mục này.


In [6]:
CHROMA_PATH = "./chroma_db"
COLLECTION_NAME = f"youtube_{VIDEO_ID}".replace("-", "_")

chroma_client = chromadb.PersistentClient(path=CHROMA_PATH)
chroma_collection = chroma_client.get_or_create_collection(COLLECTION_NAME)

vector_store = ChromaVectorStore(chroma_collection=chroma_collection)

storage_context = StorageContext.from_defaults(
    vector_store=vector_store
)

print("Chroma path:", CHROMA_PATH)
print("Collection:", COLLECTION_NAME)

Chroma path: ./chroma_db
Collection: youtube_0LCgem1OAcE


## 8. Tạo VectorStoreIndex

Cell này sẽ:

1. Chunk `documents` bằng `SentenceSplitter`
2. Tạo embedding cho từng chunk
3. Lưu embedding vào Chroma
4. Tạo `VectorStoreIndex`


In [7]:
index = VectorStoreIndex.from_documents(
    documents,
    storage_context=storage_context,
    transformations=[splitter],
    show_progress=True,
)

print("Tạo index thành công!")

Generating embeddings: 100%|██████████| 17/17 [00:01<00:00, 11.47it/s]

Tạo index thành công!


## 9. Test Retrieval

Trước khi dùng LLM để trả lời, ta nên kiểm tra hệ thống có lấy đúng đoạn liên quan không.

Đây là bước debug quan trọng trong RAG.


In [8]:
retriever = index.as_retriever(similarity_top_k=3)

question = "What is the main topic of this video?"

retrieved_nodes = retriever.retrieve(question)

print("QUESTION:", question)
print("=" * 80)

for i, node in enumerate(retrieved_nodes, start=1):
    metadata = node.node.metadata
    score = node.score

    print(f"\n[{i}] score = {score}")
    print("time:", metadata.get("start_time"))
    print("url:", metadata.get("url"))
    print("text:")
    print(node.node.text[:1000])
    print("-" * 80)

QUESTION: What is the main topic of this video?

[1] score = 0.49347002296916087
time: 14:12
url: https://www.youtube.com/watch?v=0LCgem1OAcE&t=852s
text:
least once and if they don't they've been to it 
to watch the people lying. It's run by the state   department for culture and history which takes 
pride in carrying on this strong tradition of   being home to the very best liars.  My husband has been in it, my daughter's been in 
it, they've both won so it was sort of entering   it was sort of self-defence really. Why is it 
important to you? What do you see as the value of   lying? It's really just the oral tradition of 
history and storytelling in a way that makes   it palatable for others. So something that I've 
learned in this show which is sort of I've found   surprising is how important lying is socially 
it's something I've always almost looked down on   and been like whenever anyone tells 
a lie regardless of whether or not it's a white   lie or a self-serving lie I would s

## 10. Viết hàm hỏi đáp dạng retrieval-only

Hàm này chưa dùng LLM. Nó chỉ lấy ra các đoạn liên quan nhất, kèm metadata.

Dùng để kiểm tra:

- Chunk có ổn không?
- Metadata có đúng không?
- Vector search có lấy đúng đoạn không?


In [9]:
def search_video(question: str, top_k: int = 3):
    retriever = index.as_retriever(similarity_top_k=top_k)
    nodes = retriever.retrieve(question)

    print("QUESTION:", question)
    print("=" * 80)

    for i, node in enumerate(nodes, start=1):
        metadata = node.node.metadata

        print(f"\nKết quả {i}")
        print("Score:", node.score)
        print("Time:", metadata.get("start_time"))
        print("URL:", metadata.get("url"))
        print("Text:")
        print(node.node.text[:1200])
        print("-" * 80)

    return nodes


nodes = search_video("What does the speaker explain in the video?", top_k=3)

QUESTION: What does the speaker explain in the video?

Kết quả 1
Score: 0.49961834722856596
Time: 15:45
URL: https://www.youtube.com/watch?v=0LCgem1OAcE&t=945s
Text:
stick to the truth, the less your body betrays you 
I think in telling the story. Lying is complicated.   It takes a lot of mental work to do it. You have 
to come up with an entirely new story and that's   all going on inside our brains in a way that's 
invisible to people watching. Unless you're a   neuroscientist with an MRI scanner. I'm Tali Sharot,
a professor of cognitive neuroscience at UCL. So what is going on inside our brains when we lie? Right so when we lie we need to do two things: We need to suppress the truth and then we need 
to make up something new. Can you see those two   things happening in the brain if you look inside 
it? Yeah so you'd see the frontal lobes active   both in suppression of the truth, both in you 
know imagining and inventing something new which   is a lie and then you could also see ac

## 11. Tùy chọn: sinh câu trả lời bằng LLM

Phần này cần API key hoặc local LLM.

### Cách đơn giản nếu bạn có OpenAI API key

Cài thêm:

```bash
pip install llama-index-llms-openai
```

Sau đó bỏ comment và chạy cell bên dưới.

> Nếu bạn chưa có API key thì cứ bỏ qua phần này. Phần retrieval ở trên vẫn đủ để demo RAG cơ bản.


In [ ]:
# # Bỏ comment nếu bạn có OpenAI API key
# # os.environ["OPENAI_API_KEY"] = "YOUR_API_KEY"

# from llama_index.llms.openai import OpenAI

# Settings.llm = OpenAI(model="gpt-4o-mini")

# query_engine = index.as_query_engine(
#     similarity_top_k=3,
# )

# response = query_engine.query("Summarize the video in 5 bullet points.")
# print(response)

## 12. Tùy chọn: tạo câu trả lời thủ công từ retrieved context

Nếu chưa dùng LLM, bạn vẫn có thể lấy context rồi tự đọc/kiểm tra.

Cell này gom các đoạn retrieve được thành một context block.


In [ ]:
def build_context_from_nodes(nodes):
    context_parts = []

    for i, node in enumerate(nodes, start=1):
        metadata = node.node.metadata
        context_parts.append(
            f"[Source {i}]\n"
            f"Time: {metadata.get('start_time')}\n"
            f"URL: {metadata.get('url')}\n"
            f"Content: {node.node.text}\n"
        )

    return "\n\n".join(context_parts)


question = "What are the key ideas mentioned in this video?"
nodes = retriever.retrieve(question)
context = build_context_from_nodes(nodes)

print("QUESTION:")
print(question)
print("\nCONTEXT:")
print(context[:4000])

## 13. Một số câu hỏi để test

Bạn có thể thay đổi `question` ở các cell trên bằng các câu hỏi như:

```text
What is the video mainly about?
What problem is being discussed?
What tools or technologies are mentioned?
What are the key steps explained in the video?
Can you summarize the first part of the video?
```

Nếu video tiếng Việt:

```text
Video này nói về nội dung gì?
Các ý chính trong video là gì?
Người nói đang giải thích công cụ nào?
Quy trình được trình bày trong video gồm những bước nào?
```


## 14. Ghi chú debug lỗi thường gặp

### Lỗi 1: `cannot import name 'VecterStoreIndex'`

Sai chính tả. Đúng là:

```python
VectorStoreIndex
```

### Lỗi 2: `YouTubeTranscriptApi has no attribute get_transcript`

Version mới dùng:

```python
ytt_api = YouTubeTranscriptApi()
ytt_api.fetch(video_id)
```

### Lỗi 3: truyền sai video ID

Sai:

```python
VIDEO_ID = "0LCgem1OAcE&t=881s"
```

Đúng:

```python
VIDEO_ID = "0LCgem1OAcE"
```

### Lỗi 4: LlamaIndex cố gọi OpenAI

Để test retrieval không cần LLM, dùng:

```python
Settings.llm = None
retriever = index.as_retriever()
```

Không dùng `as_query_engine()` nếu chưa cấu hình LLM.
